# Matplotlib + integración de datos del socio formador


## Propósito
Construir cuatro visualizaciones modelo que después servirán como referencia para los tres equipos:
1. Barras horizontales: inventario por marca.
2. Línea: compras frente a salidas por mes.
3. Pareto corregido: concentración de salidas por producto.
4. Dispersión: unidades compradas frente a precio mediano.

Además, se revisan `merge`, `join`, `concat`, `groupby` y agregaciones. 


## 1. Preparación del entorno y carga del libro




In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Nota: Si la carpeta 'Datos' está en el mismo lugar que tu notebook, quita .parents[2]
ARCHIVO = Path.cwd().resolve() / "Datos" / "Datos_CLV_2026_Tec_v2.xlsx"

print('Directorio actual:', Path.cwd())
print('Ruta buscada:', ARCHIVO)
print('Archivo encontrado:', ARCHIVO.exists())

if ARCHIVO.exists():
    libro = pd.ExcelFile(ARCHIVO)
    print('Hojas disponibles:')
    for hoja in libro.sheet_names:
        print(' •', hoja)
else:
    print('Revisa la ruta o la cantidad de niveles en .parents[]')

## 2. Cargar las hojas necesarias



In [ ]:
HOJA_EXIST = "Existencias_04_09_2026"
HOJA_SALIDAS = "Salidas_Almacen"
HOJA_COMPRAS = "Histórico_Compras"
HOJA_CLASIF = "Clasificación_Articulos"

exist = pd.read_excel(ARCHIVO, sheet_name=HOJA_EXIST)
sal = pd.read_excel(ARCHIVO, sheet_name=HOJA_SALIDAS)
com = pd.read_excel(ARCHIVO, sheet_name=HOJA_COMPRAS)
clas = pd.read_excel(ARCHIVO, sheet_name=HOJA_CLASIF)

for df in [exist, sal, com, clas]:
    df.columns = df.columns.astype(str).str.strip()

print("Existencias:", exist.shape)
print("Salidas:", sal.shape)
print("Compras:", com.shape)
print("Clasificación:", clas.shape)
display(exist.head(3), sal.head(3), com.head(3), clas.head(3))

## 3. Preparar tipos y claves

Los identificadores se convertirán a texto para evitar perder ceros iniciales. Las fechas y cantidades se convertirán explícitamente. La normalización no garantiza que dos claves representen el mismo producto; eso se comprueba en el siguiente paso.

In [ ]:
def normalizar_id(serie):
    return (serie.astype("string").str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA}))

for df in [exist, sal, com, clas]:
    if "articulo_id" in df.columns:
        df["articulo_id"] = normalizar_id(df["articulo_id"])

for col in ["existencias", "valor_total_inventario"]:
    if col in exist:
        exist[col] = pd.to_numeric(exist[col], errors="coerce")

for col in ["salidas"]:
    if col in sal:
        sal[col] = pd.to_numeric(sal[col], errors="coerce")

for col in ["unidades", "precio_unitario", "precio_total_neto"]:
    if col in com:
        com[col] = pd.to_numeric(com[col], errors="coerce")

for df in [sal, com]:
    if "fecha" in df:
        df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", utc=True).dt.tz_convert(None)

if "marca" in exist:
    exist["marca"] = exist["marca"].astype("string").str.strip()
if "marca" in sal:
    sal["marca"] = sal["marca"].astype("string").str.strip()

display(exist.dtypes.to_frame("tipo_existencias"))

## 4. Auditoría de claves: ¿es posible integrar las hojas?

Antes de hacer `merge`, compararemos número de claves únicas, claves compartidas y duplicados. Una coincidencia alta no basta por sí sola: también debemos revisar si la tabla que se incorpora tiene una fila por clave o varias.

In [ ]:
def resumen_claves(nombre, df):
    if "articulo_id" not in df:
        return {"hoja": nombre, "filas": len(df), "ids_unicos": 0, "ids_nulos": len(df), "ids_duplicados": 0}
    s = df["articulo_id"]
    return {
        "hoja": nombre,
        "filas": len(df),
        "ids_unicos": s.nunique(),
        "ids_nulos": int(s.isna().sum()),
        "ids_duplicados": int(s.duplicated().sum())
    }

display(pd.DataFrame([
    resumen_claves("Existencias", exist),
    resumen_claves("Salidas", sal),
    resumen_claves("Compras", com),
    resumen_claves("Clasificación", clas)
]))

def comparar_ids(nombre_a, a, nombre_b, b):
    ka = set(a["articulo_id"].dropna()) if "articulo_id" in a else set()
    kb = set(b["articulo_id"].dropna()) if "articulo_id" in b else set()
    comunes = ka & kb
    return {
        "comparacion": f"{nombre_a} ↔ {nombre_b}",
        "ids_A": len(ka), "ids_B": len(kb),
        "coincidencias": len(comunes),
        "cobertura_A": len(comunes)/len(ka) if ka else np.nan,
        "cobertura_B": len(comunes)/len(kb) if kb else np.nan
    }

auditoria = pd.DataFrame([
    comparar_ids("Existencias", exist, "Salidas", sal),
    comparar_ids("Existencias", exist, "Compras", com),
    comparar_ids("Existencias", exist, "Clasificación", clas),
    comparar_ids("Salidas", sal, "Compras", com)
])
display(auditoria.style.format({"cobertura_A":"{:.1%}", "cobertura_B":"{:.1%}"}))



**Discusión:** ¿qué pares tienen cobertura suficiente para investigar una integración? ¿Hay IDs repetidos? ¿Qué significa que compras y salidas tengan múltiples filas por producto?

No hagas un merge directo entre transacciones: un producto puede aparecer muchas veces en ambas tablas y multiplicar registros. Primero agregaremos cada tabla a una granularidad común.

**Interpretación:** existencias tiene una fila por ID en este corte; salidas y compras son transaccionales y repiten IDs; clasificación contiene varias filas por producto. No se deben unir directamente las tablas transaccionales. Tampoco se debe tratar clasificación como dimensión única si hay categorías múltiples.

La tabla de auditoría responde: **¿cuál es la granularidad de cada fuente y qué riesgo existe al unirla?**

- `Existencias`: una fila por artículo en el corte. Puede ser tabla base para análisis de inventario actual.
- `Salidas`: múltiples movimientos por artículo. Repetición esperada.
- `Compras`: múltiples operaciones por artículo. Repetición esperada.
- `Clasificación`: múltiples registros 

## 5. Primera visualización multivariable: existencias y clasificación

La clasificación funciona como una dimensión descriptiva. Para integrarla de forma segura, debe existir una sola categoría por `articulo_id`. Usaremos `validate="many_to_one"` para detectar relaciones ambiguas.

In [ ]:
clasif = clas[["articulo_id", "categoria"]].dropna(subset=["articulo_id"]).copy()
conflictos = clasif.groupby("articulo_id")["categoria"].nunique()
ids_conflictivos = conflictos[conflictos > 1].index

print("IDs con más de una categoría:", len(ids_conflictivos))
if len(ids_conflictivos) == 0:
    clasif = clasif.drop_duplicates("articulo_id")
    exist_cat = exist.merge(clasif, on="articulo_id", how="left", validate="many_to_one", indicator=True)
    print(exist_cat["_merge"].value_counts())
else:
    print("No se integra clasificación hasta resolver los IDs con categorías contradictorias.")
    exist_cat = exist.copy()

5,343 IDs con más de una categoría en la ejecución compartida. La integración se detiene deliberadamente.

La respuesta correcta no siempre es producir un `merge`: a veces es documentar por qué no se debe hacer.

## 6. Agregar salidas y compras antes de integrar

Construiremos una tabla por producto y mes para cada fuente. `groupby` define la granularidad y `agg` calcula indicadores. Solo después compararemos las dos tablas agregadas.

In [ ]:
sal_ok = sal.dropna(subset=["fecha", "articulo_id"]).copy()
com_ok = com.dropna(subset=["fecha", "articulo_id"]).copy()
sal_ok["mes"] = sal_ok["fecha"].dt.to_period("M").dt.to_timestamp()
com_ok["mes"] = com_ok["fecha"].dt.to_period("M").dt.to_timestamp()

#Métricas de salidas: Calcula el total de artículos despachados 
# (sum) y cuántas transacciones individuales hubo (size).

sal_mes = (sal_ok.groupby(["articulo_id", "mes"], as_index=False)
           .agg(unidades_salida=("salidas", "sum"),
                registros_salida=("salidas", "size")))

#Métricas de compras: Calcula el total de unidades adquiridas (sum), 
# el precio unitario típico sin sesgos por valores extremos (median) 
# y el número de registros de compra (size).

com_mes = (com_ok.groupby(["articulo_id", "mes"], as_index=False)
           .agg(unidades_compra=("unidades", "sum"),
                precio_mediano=("precio_unitario", "median"),
                registros_compra=("unidades", "size")))

print("Salidas agregadas:", sal_mes.shape)
print("Compras agregadas:", com_mes.shape)
display(sal_mes.head(), com_mes.head())

#Control de integridad (validate="one_to_one"): Asegura que no existan duplicados 
# de producto-mes en ninguna de las dos tablas antes de unirlas. Si hubiera duplicados, 
# el código lanzaría un error para evitar multiplicar filas accidentalmente.

#Rastreo de origen (indicator=True): Crea la columna _merge que etiqueta de 
# dónde proviene cada fila combinada (both, left_only o right_only).

comparacion = sal_mes.merge(com_mes, on=["articulo_id", "mes"],
                            how="outer", validate="one_to_one",
                            indicator=True)
comparacion[["unidades_salida", "unidades_compra"]] = comparacion[
    ["unidades_salida", "unidades_compra"]].fillna(0)
print(comparacion["_merge"].value_counts())

### ¿Qué significa la unión `outer`?
Conserva combinaciones producto-mes presentes en compras, en salidas o en ambas. El indicador `_merge` permite distinguir:
- `both`: hay registros en ambas fuentes para ese producto-mes.
- `left_only`: hay registro de salida, no de compra.
- `right_only`: hay registro de compra, no de salida.

Un cero tras rellenar unidades significa que no había fila de movimiento en esa fuente para esa combinación. **No demuestra que el inventario físico fuera cero.**

## 7. Ejemplo 1 — Barras horizontales: inventario por marca

**Pregunta:** ¿qué marcas concentran el valor registrado del inventario?

Cada barra representa la suma de `valor_total_inventario` de los artículos de una marca. Ordenamos de menor a mayor dentro del top 10 para que la marca de mayor valor quede arriba.

**Por qué esta gráfica:** el objetivo es comparar categorías nominales. Las barras horizontales facilitan leer nombres de marcas y comparar magnitudes. El corte compartido contiene un solo almacén (Almacén General), por lo que segmentar por almacén no aportaría comparación.

In [ ]:
inv = exist.copy()
inv["marca"] = inv["marca"].fillna("Sin marca").astype(str).str.strip()
res_marca = (inv.groupby("marca", as_index=False)
             .agg(productos=("articulo_id", "nunique"),
                  unidades=("existencias", "sum"),
                  valor=("valor_total_inventario", "sum"))
             .sort_values("valor", ascending=False))

top_valor = res_marca.head(10).sort_values("valor")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_valor["marca"], top_valor["valor"])
ax.set(title="Top 10 marcas por valor registrado de inventario",
       xlabel="Valor registrado", ylabel="Marca")
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plt.show()
display(res_marca.head(10))

### Interpretación del ejemplo 1
- La marca en la parte superior es la que tiene mayor valor agregado dentro del corte.
- Una marca puede tener mayor valor y menos unidades que otra: valor y cantidad son dimensiones distintas.
- El gráfico ayuda a priorizar una revisión, pero **no demuestra** sobreinventario, baja rotación, rentabilidad ni necesidad de compra.
- Para recomendar una acción se necesitarían consumo/demanda, antigüedad, inventario histórico, pedidos abiertos, etc...

**Preguntas**
1. ¿Qué marca tiene mayor valor? La primera fila de `res_marca`, que debe citarse con su cifra de la ejecución.
2. ¿Coincide con la marca de más unidades? Comparar `res_marca.sort_values("unidades", ascending=False)`.
3. ¿Qué significa que no coincidan? La composición de valor por unidad/producto puede ser diferente; no implica por sí sola una causa.

## 8. Ejemplo 2 — Línea: compras frente a salidas por mes

**Pregunta:** ¿cómo evolucionan las unidades de compras y salidas registradas a través del tiempo?

Cada línea representa la suma mensual de una fuente. El eje X es temporal, por eso la línea permite reconocer tendencia, picos y cambios entre meses. La agregación mensual se hace antes de graficar.


In [ ]:
resumen_mov = (comparacion.groupby("mes", as_index=False)
               [["unidades_compra", "unidades_salida"]].sum()
               .sort_values("mes"))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(resumen_mov["mes"], resumen_mov["unidades_compra"],
        marker="o", label="Unidades compradas")
ax.plot(resumen_mov["mes"], resumen_mov["unidades_salida"],
        marker="o", label="Unidades de salida")
ax.set(title="Compras y salidas registradas por mes",
       xlabel="Mes", ylabel="Unidades registradas")
ax.legend()
ax.grid(alpha=.25)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Interpretación del ejemplo 2
- Cada pico identifica un mes con un volumen registrado elevado.
- La distancia entre las líneas muestra diferencias en los movimientos registrados.
- No es un gráfico de saldo de inventario: no incorpora inventario inicial, recepciones efectivas, devoluciones, ajustes, transferencias ni todos los movimientos posibles.
- No se debe concluir que compras de un mes abastecieron las salidas del mismo mes.

**Preguntas**
1. ¿Qué mes tiene el máximo de compras? Obtenerlo con `resumen_mov.loc[resumen_mov.unidades_compra.idxmax()]`.
2. ¿Qué mes tiene el máximo de salidas? Obtenerlo de forma equivalente para `unidades_salida`.
3. ¿Qué meses divergen? Comparar las dos columnas y revisar los mayores valores absolutos de su diferencia.
4. ¿Qué dato falta? Inventario por fecha, fecha de pedido, fecha de recepción, pedidos abiertos, devoluciones/ajustes y demanda.

## 9. Ejemplo 3 — Pareto: concentración de salidas

**Pregunta:** ¿cuántos productos explican una proporción importante del total de salidas?

Cada barra representa las salidas acumuladas de un producto, ordenadas de mayor a menor. La línea representa el porcentaje acumulado del total de salidas de **todos** los productos.


In [ ]:
# Agrupación y ordenamiento: Toma los registros de salidas (sal_ok), 
# suma las unidades por cada artículo (articulo_id) 
# y los ordena estrictamente de mayor a menor demanda.

sal_producto = (sal_ok.groupby(["articulo_id", "nombre_articulo"], as_index=False)
                .agg(salidas=("salidas", "sum"))
                .sort_values("salidas", ascending=False))

#Cálculo sobre el total real: Calcula el porcentaje de participación de
#cada producto dividiendo sus salidas entre total_salidas (el universo 
#completo de productos, no solo los primeros 20). Luego obtiene el acumulado con .cumsum().

total_salidas = sal_producto["salidas"].sum()
sal_producto["porcentaje"] = 100 * sal_producto["salidas"] / total_salidas
sal_producto["porcentaje_acumulado"] = sal_producto["porcentaje"].cumsum()

# Cantidad real de productos para llegar al 80%
n80 = int((sal_producto["porcentaje_acumulado"] < 80).sum() + 1)
print("Productos necesarios para alcanzar al menos 80%:", n80)

#Construcción visual dual (twinx): Crea un gráfico con dos ejes verticales. 
# El izquierdo (ax1) dibuja las barras con las unidades físicas de los 20 productos líderes. 
# El derecho (ax2) traza la curva del porcentaje acumulado real (fijado de 0% a 100%) y 
# dibuja una línea horizontal punteada en el 80% como referencia visual.

top20 = sal_producto.head(20).copy()
x = np.arange(len(top20))
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.bar(x, top20["salidas"])
ax1.set_xticks(x)
ax1.set_xticklabels(top20["nombre_articulo"], rotation=75, ha="right")
ax1.set_ylabel("Unidades de salida")
ax1.set_title("Pareto de salidas por producto — acumulado sobre el total")
ax2 = ax1.twinx()
ax2.plot(x, top20["porcentaje_acumulado"], color="#C96B5B", marker="o")
ax2.axhline(80, linestyle="--", alpha=.7)
ax2.set_ylabel("% acumulado del total de salidas")
ax2.set_ylim(0, 100)
plt.tight_layout()
plt.show()
display(sal_producto.head(10))

### Interpretación del Pareto
En la ejecución compartida:
- El producto líder fue **MFT PU31 2M2**, con 4,340 salidas.
- El segundo fue **MFT PU33 2M2**, con 3,704.
- El tercero fue **FRESA ROTO RFID 1.0 ZI DM**, con 3,534.
- Los diez primeros sumaron aproximadamente **11.83%** de las salidas registradas.
- Se necesitaron **515 productos** para alcanzar al menos el 80% acumulado.

**Conclusión:** en esta base no se observa una concentración extrema del tipo “pocos productos explican 80%”; se requieren cientos de productos. El Pareto es útil precisamente para comprobarlo, no para imponer la regla 80/20.

**Preguntas**
1. ¿Qué producto lidera? MFT PU31 2M2, 4,340 salidas en el corte ejecutado.
2. ¿Cuánto representan los diez primeros? Aproximadamente 11.83%.
3. ¿Se cumple 80/20? No: hacen falta 515 productos para llegar a 80%.
4. ¿Qué decisión se deriva? Se puede segmentar la priorización por criticidad, valor o familia, pero las salidas por sí solas no bastan para determinar compras.

## 10. Ejemplo 4 — Dispersión: unidades compradas vs. precio mediano

**Pregunta:** ¿se observa una asociación descriptiva entre el volumen comprado y el precio unitario mediano por producto?

Cada punto representa un producto. El eje X suma unidades compradas; el eje Y calcula la mediana del precio unitario de sus operaciones. La mediana reduce la influencia de operaciones con precios extremos, aunque no elimina diferencias de proveedor, fecha o presentación.

In [ ]:
# Agrupación por producto: Toma la base de compras (com_ok) y resume cada artículo
#  calculando tres métricas clave: el total de unidades compradas (sum), 
# el precio unitario mediano (median) y la frecuencia o número de operaciones (count de folios).

compra_producto = (com_ok.groupby(["articulo_id", "nombre_articulo"], as_index=False)
                   .agg(unidades=("unidades", "sum"),
                        precio_mediano=("precio_unitario", "median"),
                        operaciones=("folio", "count")))

#Filtro de recurrencia (operaciones >= 2): Descarta los artículos que solo se compraron una vez
#  y aquellos sin precio válido. Esto asegura que el gráfico muestre únicamente productos con historial repetitivo de abastecimiento.

puntos = compra_producto[
    (compra_producto["operaciones"] >= 2) &
    compra_producto["precio_mediano"].notna()
].copy()

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(puntos["unidades"], puntos["precio_mediano"], alpha=.55)
ax.set(title="Unidades compradas vs. precio unitario mediano por producto",
       xlabel="Unidades compradas acumuladas",
       ylabel="Precio unitario mediano")
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()
display(puntos.sort_values("unidades", ascending=False).head(10))

### Interpretación del ejemplo 4
- Cada punto es un producto, no una compra individual.
- Puntos a la derecha representan productos con mayor volumen comprado acumulado.
- Puntos más arriba tienen precio mediano mayor.
- Una nube descendente podría sugerir asociación negativa; una nube sin patrón claro sugiere asociación débil. Se debe observar la gráfica y, si se calcula correlación, interpretarla como descriptiva.
- No se puede afirmar que comprar más reduce el precio: puede haber mezcla de productos de distinta naturaleza, proveedores, fechas, unidades de medida y condiciones comerciales.
- Esquina inferior derecha (Alto volumen / Bajo precio): Son consumibles, tornillería, refacciones menores o material de alta rotación. Se compran por cientos o miles porque son económicos y se gastan rápido.
-2. Esquina superior izquierda (Bajo volumen / Alto precio): Son equipos, refacciones críticas o herramientas especializadas. Se compran muy pocas unidades debido a su alto costo unitario.

**Preguntas**
1. ¿Hay relación clara? Responder según la nube; no asumirla.
2. ¿Qué puntos son atípicos? Identificar productos alejados del conjunto y revisar sus operaciones.
3. ¿Por qué mediana? Es menos sensible que la media a valores extremos.
4. ¿Qué faltaría? Proveedor, fecha, moneda, unidad de medida, descuentos, volumen por orden y comparación del mismo producto.